# 04 — Seeds, MuRIL and ensembles (Roman Urdu text module)

**Goal:** raise accuracy and find out which differences are real.

| Part | What | Why |
|---|---|---|
| A | Character + word TF-IDF + Logistic Regression | Strong, fast baseline that handles Roman Urdu spelling variation |
| B | XLM-R base, **3 seeds** (same settings as the Colab run) | Mean ± std, so we know how much one run varies |
| C | **MuRIL** base, 3 seeds | Model trained on South Asian languages incl. Urdu, also on transliterated text |
| D | Ensembles: average of seeds, XLM-R + MuRIL, + TF-IDF (weight chosen on validation) | Usually adds 1–2 points |
| E | Results per source + bootstrap confidence intervals | Is the best model really better than TF-IDF? |

Same fixed split as `01_make_splits.ipynb` (`split_ids.csv`), so everything is comparable with the Colab run (0.833 macro-F1).

**Runtime on the RTX A4000:** roughly 60–80 minutes in total.
**Safe to stop and restart:** each finished run saves its predictions; on a rerun, finished runs are skipped.
**No article text is saved** — output files contain ids and probabilities only.

## 0. Settings

In [11]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from pathlib import Path
DATA_PATH      = Path(r"E:\KhabarNazar_text\data\khabarnazar_roman_urdu_dataset.csv")
SPLIT_IDS_PATH = Path(r"E:\KhabarNazar_text\data\split_ids.csv")
OUT            = Path(r"E:\KhabarNazar_text\outputs\04_seeds_muril_ensemble")
OUT.mkdir(parents=True, exist_ok=True)

MODELS = {"xlmr": "xlm-roberta-base", "muril": "google/muril-base-cased"}
SEEDS  = [42, 13, 7]

# Same training settings as the Colab run (02_train_xlmr.ipynb)
MAX_LEN, BATCH_SIZE, GRAD_ACCUM = 512, 16, 1     # effective batch 16
LR, EPOCHS, WARMUP_RATIO, WEIGHT_DECAY, PATIENCE = 2e-5, 5, 0.1, 0.01, 2

LABEL2ID = {"Real": 0, "Fake": 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
print("Data:", DATA_PATH.exists(), "| split:", SPLIT_IDS_PATH.exists(), "| output folder:", OUT)

Data: True | split: True | output folder: E:\KhabarNazar_text\outputs\04_seeds_muril_ensemble


## 1. GPU check

In [12]:
import torch, transformers, sklearn, platform
print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
print("transformers", transformers.__version__, "| sklearn", sklearn.__version__)
NUM_WORKERS = 0 if platform.system() == "Windows" else 2

PyTorch 2.11.0+cu128 | CUDA: True | NVIDIA RTX A4000
transformers 5.20.2 | sklearn 1.9.1


## 2. Load data and the fixed split

In [13]:
import numpy as np, pandas as pd

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")[["id", "roman_text", "label", "source_dataset"]]
df = df.rename(columns={"roman_text": "text", "source_dataset": "source"})
df["text"] = df["text"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
df["label"] = df["label"].astype(str).str.strip().str.capitalize().map(LABEL2ID)
assert df["label"].notna().all()
df["label"] = df["label"].astype(int)

df = df.merge(pd.read_csv(SPLIT_IDS_PATH), on="id", how="left", validate="one_to_one")
assert df["split"].notna().all()
train_df = df[df.split == "train"].reset_index(drop=True)
val_df   = df[df.split == "val"].reset_index(drop=True)
test_df  = df[df.split == "test"].reset_index(drop=True)
print("train / val / test:", len(train_df), len(val_df), len(test_df), "(expected 6615 / 826 / 826)")
SOURCES = sorted(df.source.unique())

train / val / test: 6615 826 826 (expected 6615 / 826 / 826)


## 3. Evaluation helpers

In [14]:
from sklearn.metrics import accuracy_score, f1_score

def scores(frame, p_fake, thr=0.5):
    # macro-F1 overall and per source, plus overall accuracy
    pred = (np.asarray(p_fake) >= thr).astype(int)
    y = frame["label"].to_numpy()
    out = {"accuracy": accuracy_score(y, pred), "ALL": f1_score(y, pred, average="macro")}
    for s in SOURCES:
        m = (frame["source"] == s).to_numpy()
        out[s] = f1_score(y[m], pred[m], average="macro")
    return out

def save_probs(tag, val_p, test_p):
    pd.DataFrame({"id": val_df.id, "split": "val", "p_fake": val_p}).to_csv(OUT / f"probs_{tag}_val.csv", index=False)
    pd.DataFrame({"id": test_df.id, "split": "test", "p_fake": test_p}).to_csv(OUT / f"probs_{tag}_test.csv", index=False)

def load_probs(tag):
    v, t = OUT / f"probs_{tag}_val.csv", OUT / f"probs_{tag}_test.csv"
    if v.exists() and t.exists():
        v, t = pd.read_csv(v), pd.read_csv(t)
        assert (v.id.values == val_df.id.values).all() and (t.id.values == test_df.id.values).all()
        return v.p_fake.to_numpy(), t.p_fake.to_numpy()
    return None

## A. TF-IDF baseline (character + word n-grams)

Character n-grams cope with Roman Urdu spelling variation (*nahi / nahin / nai*). `C` is chosen on the validation set.

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.linear_model import LogisticRegression

def make_tfidf(C):
    feats = make_union(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, lowercase=True),
        TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=True),
    )
    return make_pipeline(feats, LogisticRegression(C=C, max_iter=3000, class_weight="balanced"))

if load_probs("tfidf") is None:
    best = None
    for C in [0.5, 1, 2, 5, 10, 20]:
        clf = make_tfidf(C).fit(train_df.text, train_df.label)
        f = scores(val_df, clf.predict_proba(val_df.text)[:, 1])["ALL"]
        print(f"C={C:<4} val macro-F1 {f:.4f}")
        if best is None or f > best[0]:
            best = (f, C, clf)
    print("Best C:", best[1])
    clf = best[2]
    save_probs("tfidf", clf.predict_proba(val_df.text)[:, 1], clf.predict_proba(test_df.text)[:, 1])
else:
    print("TF-IDF predictions already saved — skipped")

TF-IDF predictions already saved — skipped


## B + C. Transformer runs (XLM-R and MuRIL, 3 seeds each)

Each run: fine-tune up to 5 epochs, keep the epoch with the best validation macro-F1, save validation and test probabilities, then delete the checkpoints (to save disk space).

In [16]:
import gc, math, shutil, time
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          TrainingArguments, Trainer, EarlyStoppingCallback, set_seed)

def hf_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds), "f1_macro": f1_score(labels, preds, average="macro")}

def run_transformer(key, seed):
    tag = f"{key}_s{seed}"
    if load_probs(tag) is not None:
        print(f"{tag}: already done — skipped"); return
    print(f"\n===== {tag} ({MODELS[key]}) =====")
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(MODELS[key])
    enc = lambda d: Dataset.from_pandas(d[["text", "label"]]).map(
        lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=["text"])
    tr, va, te = enc(train_df), enc(val_df), enc(test_df)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODELS[key], num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID)
    steps = math.ceil(len(tr) / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
    ckpt = OUT / f"checkpoints_{tag}"
    args = TrainingArguments(
        output_dir=str(ckpt), num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE * 2,
        gradient_accumulation_steps=GRAD_ACCUM, warmup_steps=int(WARMUP_RATIO * steps),
        weight_decay=WEIGHT_DECAY, lr_scheduler_type="linear",
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, save_only_model=True,
        load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
        logging_steps=50, bf16=torch.cuda.is_bf16_supported(), dataloader_num_workers=NUM_WORKERS,
        report_to="none", seed=seed)
    trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=va,
                      processing_class=tok, data_collator=DataCollatorWithPadding(tok),
                      compute_metrics=hf_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    t0 = time.time()
    trainer.train()
    prob = lambda ds: torch.softmax(torch.tensor(trainer.predict(ds).predictions).float(), -1)[:, 1].numpy()
    val_p, test_p = prob(va), prob(te)
    save_probs(tag, val_p, test_p)
    pd.DataFrame(trainer.state.log_history).to_csv(OUT / f"log_{tag}.csv", index=False)
    print(f"{tag}: {(time.time()-t0)/60:.1f} min | val F1 {scores(val_df, val_p)['ALL']:.4f} | test F1 {scores(test_df, test_p)['ALL']:.4f}")

    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(ckpt, ignore_errors=True)

### B. XLM-R, 3 seeds (~10 min each)

In [17]:
for s in SEEDS:
    run_transformer("xlmr", s)

xlmr_s42: already done — skipped
xlmr_s13: already done — skipped
xlmr_s7: already done — skipped


### C. MuRIL, 3 seeds (~10 min each)

If MuRIL fails to download or load, skip this cell — the results section works with whatever runs exist.

In [18]:
   for s in [42, 21, 7]:
       run_transformer("muril", s)

muril_s42: already done — skipped

===== muril_s21 (google/muril-base-cased) =====


Map:   0%|          | 0/6615 [00:00<?, ? examples/s]

Map:   0%|          | 0/826 [00:00<?, ? examples/s]

Map:   0%|          | 0/826 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

muril_s21: 10.4 min | val F1 0.8095 | test F1 0.8073
muril_s7: already done — skipped


## D + E. Results: single runs, seed averages, ensembles

In [19]:
P = {}   # tag -> (val_probs, test_probs)
for tag in ["tfidf"] + [f"xlmr_s{s}" for s in SEEDS] + [f"muril_s{s}" for s in [42, 21, 7]]:
    p = load_probs(tag)
    if p is not None:
        P[tag] = p
print("Available runs:", list(P))

rows = []
def add_row(name, val_p, test_p, note=""):
    r = scores(test_df, test_p); r["model"] = name; r["val_ALL"] = scores(val_df, val_p)["ALL"]; r["note"] = note
    rows.append(r)

if "tfidf" in P:
    add_row("TF-IDF char+word + LR", *P["tfidf"])

for key in MODELS:
    tags = [t for t in P if t.startswith(key + "_")]
    if not tags: continue
    per_seed = [scores(test_df, P[t][1]) for t in tags]
    mean = {c: np.mean([r[c] for r in per_seed]) for c in per_seed[0]}
    std  = {c: np.std([r[c] for r in per_seed], ddof=1) if len(per_seed) > 1 else 0.0 for c in per_seed[0]}
    for t in tags:
        add_row(f"{key} single run ({t.split('_')[1]})", *P[t])
    rows.append({**mean, "model": f"{key} mean of {len(tags)} seeds",
                 "note": f"std ALL ±{std['ALL']:.3f}, BtT ±{std['bend_the_truth']:.3f}, FIRE ±{std['fire']:.3f}"})
    add_row(f"{key} ensemble ({len(tags)} seeds)",
            np.mean([P[t][0] for t in tags], 0), np.mean([P[t][1] for t in tags], 0))

def ens(prefixes):
    tags = [t for t in P if any(t.startswith(p + "_") for p in prefixes)]
    return (np.mean([P[t][0] for t in tags], 0), np.mean([P[t][1] for t in tags], 0)) if tags else None

both = ens(["xlmr", "muril"])
if both is not None and any(t.startswith("muril_") for t in P) and any(t.startswith("xlmr_") for t in P):
    add_row("XLM-R + MuRIL ensemble", *both)

# Transformer ensemble + TF-IDF, weight chosen on validation only
best_tr = both if both is not None else None
if best_tr is not None and "tfidf" in P:
    grid = np.linspace(0, 1, 21)
    w = max(grid, key=lambda w: scores(val_df, (1-w)*best_tr[0] + w*P["tfidf"][0])["ALL"])
    add_row(f"Transformers + TF-IDF (w_tfidf={w:.2f})",
            (1-w)*best_tr[0] + w*P["tfidf"][0], (1-w)*best_tr[1] + w*P["tfidf"][1], "weight chosen on validation")

cols = ["model", "accuracy", "ALL"] + SOURCES + ["val_ALL", "note"]
results = pd.DataFrame(rows)[cols]
results.to_csv(OUT / "results_04.csv", index=False)
print("Test macro-F1 (ALL and per source) on the fixed 826-article test split.")
print("Reference: Colab XLM-R 0.833 | Colab TF-IDF 0.820 | run 2 0.825")
pd.set_option("display.width", 200)
results.round(3)

Available runs: ['tfidf', 'xlmr_s42', 'xlmr_s13', 'xlmr_s7', 'muril_s42', 'muril_s21', 'muril_s7']
Test macro-F1 (ALL and per source) on the fixed 826-article test split.
Reference: Colab XLM-R 0.833 | Colab TF-IDF 0.820 | run 2 0.825


,model,accuracy,ALL,ax_to_grind,bend_the_truth,fire,hook_and_bait,val_ALL,note
0,TF-IDF char+word + LR,0.816,0.816,0.801,0.658,0.620,0.883,0.819,
1,xlmr single run (s42),0.844,0.844,0.814,0.639,0.782,0.909,0.839,
2,xlmr single run (s13),0.823,0.823,0.822,0.584,0.627,0.895,0.819,
3,xlmr single run (s7),0.814,0.812,0.784,0.584,0.676,0.886,0.825,
4,xlmr mean of 3 seeds,0.827,0.826,0.806,0.602,0.695,0.897,NaN,"std ALL ±0.016, BtT ±0.032, FIRE ±0.079"
5,xlmr ensemble (3 seeds),0.837,0.836,0.818,0.614,0.688,0.907,0.844,
6,muril single run (s42),0.835,0.835,0.846,0.584,0.720,0.893,0.829,
7,muril single run (s21),0.808,0.807,0.810,0.607,0.663,0.857,0.810,
8,muril single run (s7),0.812,0.811,0.799,0.584,0.662,0.881,0.823,
9,muril mean of 3 seeds,0.818,0.818,0.818,0.592,0.682,0.877,NaN,"std ALL ±0.015, BtT ±0.013, FIRE ±0.033"


### Is the best model really better than TF-IDF?

Paired bootstrap over the 826 test articles: resample the test set 2,000 times and see how often the difference stays above zero. If the 95% interval includes 0, the difference is not reliable.

In [20]:
def bootstrap_diff(p_a, p_b, frame, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    y = frame.label.to_numpy(); a = (p_a >= .5).astype(int); b = (p_b >= .5).astype(int)
    diffs = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        diffs.append(f1_score(y[i], a[i], average="macro") - f1_score(y[i], b[i], average="macro"))
    return np.mean(diffs), np.percentile(diffs, [2.5, 97.5])

comparisons = {}
for key in MODELS:
    e = ens([key])
    if e is not None: comparisons[f"{key} ensemble"] = e[1]
if both is not None: comparisons["XLM-R + MuRIL ensemble"] = both[1]

if "tfidf" in P:
    for name, p in comparisons.items():
        d, (lo, hi) = bootstrap_diff(p, P["tfidf"][1], test_df)
        verdict = "better" if lo > 0 else ("worse" if hi < 0 else "not clearly different")
        print(f"{name:26s} vs TF-IDF: {d:+.3f} macro-F1 (95% CI {lo:+.3f} to {hi:+.3f}) -> {verdict}")

xlmr ensemble              vs TF-IDF: +0.020 macro-F1 (95% CI -0.006 to +0.049) -> not clearly different
muril ensemble             vs TF-IDF: +0.014 macro-F1 (95% CI -0.012 to +0.042) -> not clearly different
XLM-R + MuRIL ensemble     vs TF-IDF: +0.021 macro-F1 (95% CI -0.007 to +0.048) -> not clearly different


## Notes for the report

- Report the **mean ± std over seeds** for single models, not the best seed.
- Bend the Truth (89 test articles) and FIRE (70) have about ±10 points of uncertainty; treat their differences with care.
- Ensembles were combined with weights chosen on validation only; the test set was used once, at the end.